# 🔌 YOLOv11n — Smart Electricity Meter Digit Detection

**Mục tiêu:** Fine-tune YOLOv11n để phát hiện vùng chữ số trên mặt công tơ điện.

**Pipeline:** `YOLOv11n (detect vùng số)` → `PaddleOCR (nhận diện ký tự)`

**Kaggle Settings:**
- Runtime: **GPU T4 x2** hoặc **GPU P100**
- Internet: **ON**
- Persistence: **ON**

## 1️⃣ Cài đặt thư viện

In [ ]:
# Gỡ ray để tránh xung đột với ultralytics trên Kaggle
!pip uninstall -y ray 2>/dev/null
!pip install -q ultralytics==8.3.40

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 2️⃣ Load Dataset

Dataset đã được chuẩn bị sẵn tại `training/dataset/` (YOLO format).

**Trên Kaggle:** Upload thư mục `training/dataset` lên Kaggle Dataset, rồi đổi `DATASET_DIR` bên dưới.

| Split | Images |
|-------|--------|
| Train | 60 |
| Valid | 15 |
| Test  | 7 |

In [ ]:
import os

# === Local Dataset ===
# Trên Kaggle: đổi thành path của Kaggle Dataset, ví dụ:
# DATASET_DIR = "/kaggle/input/smart-meter-dataset"
DATASET_DIR = os.path.abspath(os.path.join(os.getcwd(), "dataset"))
DATA_YAML = os.path.join(DATASET_DIR, "data.yaml")

assert os.path.exists(DATA_YAML), f"❌ Không tìm thấy {DATA_YAML}"
print(f"✅ Dataset: {DATASET_DIR}")
print(f"📄 Config:  {DATA_YAML}")

## 3️⃣ Kiểm tra Dataset

In [ ]:
import glob
import os
import yaml
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Đọc data.yaml
with open(DATA_YAML) as f:
    data_config = yaml.safe_load(f)

print("📋 Dataset Config:")
print(f"  Classes: {data_config.get('names', {})}")
print(f"  Train path: {data_config.get('train')}")
print(f"  Val path: {data_config.get('val')}")

# Đếm ảnh
base = DATASET_DIR
for split in ['train', 'val', 'valid', 'test']:
    imgs = glob.glob(f"{base}/{split}/images/*")
    lbls = glob.glob(f"{base}/{split}/labels/*")
    if imgs:
        print(f"  {split}: {len(imgs)} images, {len(lbls)} labels")

In [ ]:
# Hiển thị mẫu ảnh + bounding box
def show_sample(img_path, label_path, class_names):
    img = Image.open(img_path)
    w, h = img.size
    fig, ax = plt.subplots(1, figsize=(8, 6))
    ax.imshow(img)

    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                xc, yc, bw, bh = [float(x) for x in parts[1:5]]
                x1 = (xc - bw/2) * w
                y1 = (yc - bh/2) * h
                rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                    linewidth=2, edgecolor='lime', facecolor='none')
                ax.add_patch(rect)
                name = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
                ax.text(x1, y1-5, name, color='lime', fontsize=10,
                    bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

    ax.set_title(os.path.basename(img_path))
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# Normalize names: data.yaml có thể trả list hoặc dict
raw_names = data_config.get('names', ['meter_digits'])
names = raw_names if isinstance(raw_names, list) else list(raw_names.values())

# Hiển thị 4 mẫu đầu tiên
train_imgs = sorted(glob.glob(f"{base}/train/images/*"))[:4]

for img_path in train_imgs:
    fname = os.path.splitext(os.path.basename(img_path))[0]
    label_path = f"{base}/train/labels/{fname}.txt"
    show_sample(img_path, label_path, names)

## 4️⃣ Training YOLOv11n

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv11n
model = YOLO("yolo11n.pt")

# Train
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,           # Giảm nếu OOM: 8
    patience=15,         # Early stopping
    device=0,
    project="/kaggle/working/runs",
    name="meter-digits",
    exist_ok=True,

    # Augmentation tối ưu cho công tơ điện
    hsv_h=0.01,         # Biến đổi màu nhẹ
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=5.0,         # Xoay nhẹ (công tơ gần như thẳng)
    translate=0.1,
    scale=0.3,
    flipud=0.0,          # Không lật dọc (số bị ngược)
    fliplr=0.0,          # Không lật ngang (số bị gương)
    mosaic=0.8,
    mixup=0.1,

    # Optimizer
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,

    # Misc
    workers=2,
    seed=42,
    verbose=True,
)

## 5️⃣ Đánh giá kết quả

In [ ]:
# Metrics
metrics = model.val()
print(f"\n📊 Evaluation Results:")
print(f"  mAP50:    {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall:    {metrics.box.mr:.4f}")

In [ ]:
# Hiển thị training curves
from IPython.display import display, Image as IPImage

run_dir = "/kaggle/working/runs/meter-digits"

for img_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.png"]:
    img_path = f"{run_dir}/{img_name}"
    if os.path.exists(img_path):
        print(f"\n📈 {img_name}")
        display(IPImage(filename=img_path, width=800))

## 6️⃣ Test trên ảnh mới

In [ ]:
# Load best model
best_model = YOLO(f"{run_dir}/weights/best.pt")

# Test trên validation set
test_imgs = sorted(glob.glob(f"{base}/valid/images/*") or glob.glob(f"{base}/val/images/*"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for idx, img_path in enumerate(test_imgs):
    ax = axes[idx // 3][idx % 3]
    results = best_model.predict(img_path, conf=0.3, verbose=False)
    annotated = results[0].plot()
    ax.imshow(annotated[:, :, ::-1])  # BGR -> RGB
    ax.set_title(f"{os.path.basename(img_path)}\n{len(results[0].boxes)} detections", fontsize=9)
    ax.axis('off')

for idx in range(len(test_imgs), 6):
    axes[idx // 3][idx % 3].axis('off')

plt.suptitle("YOLOv11n — Meter Digit Detection Results", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7️⃣ Export model

In [ ]:
import shutil

# Copy best.pt
src = f"{run_dir}/weights/best.pt"
dst = "/kaggle/working/meter_yolo11n_best.pt"
shutil.copy2(src, dst)
size_mb = os.path.getsize(dst) / 1024 / 1024
print(f"✅ Model saved: {dst} ({size_mb:.1f} MB)")

# Export ONNX (optional — cho edge deployment)
# best_model.export(format="onnx", imgsz=640, simplify=True)

print("\n📌 Hướng dẫn sử dụng:")
print("1. Download file meter_yolo11n_best.pt")
print("2. Copy vào backend/ml_models/")
print("3. Đổi YOLO_MODEL_PATH trong .env:")
print("   YOLO_MODEL_PATH=ml_models/meter_yolo11n_best.pt")

## 8️⃣ Test Pipeline đầy đủ (YOLO + OCR)

In [ ]:
# Cell 8a: Cài đặt PaddleOCR (chạy 1 lần duy nhất)
# Nếu gặp lỗi "PDX has already been initialized", restart kernel rồi chạy lại từ cell 1.
!pip install -q paddlepaddle paddleocr langchain-text-splitters

In [ ]:
# Cell 8b: Import + Init PaddleOCR (safe to re-run)
import cv2
import numpy as np
import re

# Guard: chỉ init PaddleOCR nếu chưa tồn tại — tránh lỗi "PDX already initialized"
if 'ocr' not in globals():
    from paddleocr import PaddleOCR
    ocr = PaddleOCR(use_angle_cls=False, lang='en', show_log=False)
    print("✅ PaddleOCR initialized")
else:
    print("♻️ PaddleOCR already loaded, reusing existing instance")

In [ ]:
# Cell 8c: Full pipeline test
def full_pipeline(img_path, yolo_model, ocr_engine, conf=0.3):
    """YOLO detect → crop → PaddleOCR recognize."""
    img = cv2.imread(img_path)
    results = yolo_model.predict(img, conf=conf, verbose=False)[0]

    readings = []
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        crop = img[y1:y2, x1:x2]

        # Preprocess crop
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        enhanced = cv2.convertScaleAbs(gray, alpha=1.5, beta=20)

        # OCR
        ocr_result = ocr_engine.ocr(enhanced, cls=False)
        if ocr_result and ocr_result[0]:
            for line in ocr_result[0]:
                text = line[1][0]
                confidence = line[1][1]
                digits = re.sub(r'[^0-9.]', '', text)
                if digits and confidence > 0.5:
                    readings.append({
                        'text': digits,
                        'confidence': confidence,
                        'yolo_conf': float(box.conf[0]),
                        'bbox': [x1, y1, x2, y2]
                    })
    return readings

# Test
if test_imgs:
    for img_path in test_imgs[:3]:
        readings = full_pipeline(img_path, best_model, ocr)
        print(f"\n📸 {os.path.basename(img_path)}:")
        for r in readings:
            print(f"  → {r['text']} (OCR: {r['confidence']:.1%}, YOLO: {r['yolo_conf']:.1%})")
        if not readings:
            print("  ⚠️ Không detect được số")

---

## 📝 Ghi chú

| Thông số | Giá trị khuyến nghị |
|----------|--------------------|
| Dataset | 200-500 ảnh (min 100) |
| Epochs | 100 (early stop 15) |
| Image size | 640x640 |
| Batch size | 16 (giảm 8 nếu OOM) |
| mAP50 target | > 0.90 |

### Tips:
- Chụp ảnh đa dạng: sáng/tối, góc nghiêng, bẩn/sạch
- Annotate chính xác vùng hiển thị số
- Nếu mAP < 0.85: thêm ảnh, kiểm tra labels
- Flip bị tắt vì số sẽ bị đảo ngược